# Interaktive DynaMix-Testtrajektorien

Dieses Notebook laedt DynaMix-Runs aus `results`-Ordnern oder einzelne Run-Ordner und plottet Ground Truth gegen Forecast als interaktive Plotly-3D-Trajektorie. Es kann ausserdem das originale pretrained DynaMix-Modell laden, analog zu `pretrained_vs_phi_median_slides_server.py`.

Passe nur die Konfigurationszelle direkt unterhalb an und fuehre danach die restlichen Zellen aus.

In [1]:
from pathlib import Path

# ---------------------------------------------------------------------------
# Modelle
# ---------------------------------------------------------------------------
# LOCAL_MODEL_PATHS akzeptiert:
# - einen konkreten Run-Ordner mit config.json und checkpoints/final_model.*
# - einen Repo-Ordner, der einen results-Unterordner enthaelt
# - direkt einen results-Unterordner; darin werden alle Runs rekursiv gesucht
LOCAL_MODEL_PATHS = [
    # r"C:\\Users\\koray\\OneDrive\\Dokumente\\Studium\\Master\\Heidelberg\\Master_Thesis\\DynaMix-python-b-tipping\\results",
    # r"C:\\Users\\koray\\OneDrive\\Dokumente\\Studium\\Master\\Heidelberg\\Master_Thesis\\DynaMix_context_phi\\results\\mein_run",
]

# Falls ein Run ausserhalb seines Repos liegt oder die Repo-Erkennung nicht klappt,
# trage hier optional den passenden Repo-Root ein. Wenn None, wird er inferiert.
DEFAULT_LOCAL_REPO_ROOT = None  # z.B. r"C:\\...\\DynaMix-python-b-tipping"

# Optional: originales pretrained DynaMix-Modell aus dem originalen Repo laden.
USE_PRETRAINED_HF = False
HF_REPO_ROOT = r"C:\\Users\\koray\\OneDrive\\Dokumente\\Studium\\Master\\Heidelberg\\Master_Thesis\\DynaMix-python"
HF_MODEL_NAME = "dynamix-3d-alrnn-v1.0"
HF_FORECAST_PREPROCESSING_METHOD = None  # z.B. "pos_embedding" oder None
HF_STANDARDIZE = None  # None nutzt Forecaster-Default; sonst True/False

# ---------------------------------------------------------------------------
# Testdaten
# ---------------------------------------------------------------------------
# Pro Datensatz kannst du mehrere Trajektorien angeben. Kontext startet bei
# context_start und hat context_steps Punkte. Wenn context_steps None ist,
# wird context.npy-Laenge genutzt, sonst 20% der Trajektorie.
DATASETS = [
    {
        "name": "burke_shaw_values_8",
        "path": r"C:\\Users\\koray\\OneDrive\\Dokumente\\Studium\\Master\\Heidelberg\\Master_Thesis\\generate_trainingdata\\diffrentbutconst_prams\\single_new_regimes_const\\windows\\burke_shaw_nu_15_to_30\\values_8",
        "trajectory_indices": [0],
        "context_start": 0,
        "context_steps": None,
        "forecast_steps": None,  # None = bis zum Ende der Trajektorie
    },
    # Weitere Datensaetze einfach als weitere Dicts anfuegen.
]

# ---------------------------------------------------------------------------
# Runtime / Plot
# ---------------------------------------------------------------------------
DEVICE = "auto"  # "auto", "cuda", "cuda:0" oder "cpu"
PLOT_MAX_POINTS = 12000  # Downsampling nur fuer den Plot; Forecast bleibt voll.
SHOW_CONTEXT_POINTS = True
PLOT_HEIGHT = 760
DRY_RUN = False  # True prueft nur, welche Modelle/Daten gefunden werden.

## Hilfsfunktionen

In [2]:
import inspect
import json
import sys
from dataclasses import dataclass
from typing import Any

import numpy as np
import plotly.graph_objects as go
import torch

import constant_parameter_stochastic_median_slides_server as slides


@dataclass(frozen=True)
class LocalRunSpec:
    label: str
    run_dir: Path
    repo_root: Path


@dataclass(frozen=True)
class PretrainedSpec:
    label: str
    repo_root: Path
    model_name: str


def as_path(value: str | Path | None) -> Path | None:
    return None if value is None else Path(value).expanduser().resolve()


def resolve_device(device: str) -> str:
    if device == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    if device.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError(
            f"CUDA angefragt ({device}), aber torch.cuda.is_available() ist False"
        )
    return device


def is_run_dir(path: Path) -> bool:
    return (path / "config.json").exists() and (
        (path / "checkpoints" / "final_model.safetensors").exists()
        or (path / "checkpoints" / "final_model.pt").exists()
    )


def find_runs(path: Path) -> list[Path]:
    path = path.resolve()
    if is_run_dir(path):
        return [path]
    search_root = path / "results" if (path / "results").exists() else path
    runs = sorted(
        {p.parent for p in search_root.rglob("config.json") if is_run_dir(p.parent)}
    )
    return runs


def infer_repo_root(
    run_or_results_path: Path, default_repo_root: str | Path | None = None
) -> Path:
    if default_repo_root is not None:
        repo = as_path(default_repo_root)
        if repo is None:
            raise ValueError("default_repo_root resolved to None")
        if not (repo / "src" / "dynamix").exists():
            raise FileNotFoundError(
                f"Kein src/dynamix im DEFAULT_LOCAL_REPO_ROOT gefunden: {repo}"
            )
        return repo

    path = run_or_results_path.resolve()
    candidates = [path, *path.parents]
    for candidate in candidates:
        if (candidate / "src" / "dynamix").exists():
            return candidate
        if (
            candidate.name == "results"
            and (candidate.parent / "src" / "dynamix").exists()
        ):
            return candidate.parent
    for candidate in candidates:
        if candidate.name == "results":
            return candidate.parent
    raise FileNotFoundError(
        f"Konnte Repo-Root fuer {path} nicht inferieren. Setze DEFAULT_LOCAL_REPO_ROOT."
    )


def build_model_specs() -> list[LocalRunSpec | PretrainedSpec]:
    specs: list[LocalRunSpec | PretrainedSpec] = []
    for raw_path in LOCAL_MODEL_PATHS:
        base = as_path(raw_path)
        if base is None:
            continue
        runs = find_runs(base)
        if not runs:
            raise FileNotFoundError(f"Keine Runs unter {base} gefunden")
        for run_dir in runs:
            repo_root = infer_repo_root(run_dir, DEFAULT_LOCAL_REPO_ROOT)
            try:
                rel_label = str(run_dir.relative_to(repo_root / "results"))
            except ValueError:
                rel_label = run_dir.name
            specs.append(
                LocalRunSpec(label=rel_label, run_dir=run_dir, repo_root=repo_root)
            )

    if USE_PRETRAINED_HF:
        specs.append(
            PretrainedSpec(
                label=f"pretrained:{HF_MODEL_NAME}",
                repo_root=as_path(HF_REPO_ROOT),
                model_name=HF_MODEL_NAME,
            )
        )
    return specs


def put_repo_src_first(repo_root: Path) -> None:
    src = repo_root / "src"
    if not src.exists():
        raise FileNotFoundError(f"Expected DynaMix src directory at {src}")
    src_str = str(src)
    while src_str in sys.path:
        sys.path.remove(src_str)
    sys.path.insert(0, src_str)


def load_pretrained_model(spec: PretrainedSpec, device: str):
    slides.purge_dynamix_modules()
    put_repo_src_first(spec.repo_root)
    from dynamix.model.forecaster import DynaMixForecaster
    from dynamix.utilities.utilities import load_hf_model

    model = load_hf_model(spec.model_name).to(device)
    model.eval()
    return model, DynaMixForecaster(model)


def load_any_model(spec: LocalRunSpec | PretrainedSpec, device: str):
    if isinstance(spec, PretrainedSpec):
        return load_pretrained_model(spec, device)
    return slides.load_model(spec.run_dir, device, repo_root=spec.repo_root)[:2]


def load_dataset(dataset_cfg: dict[str, Any]) -> dict[str, Any]:
    data_dir = as_path(dataset_cfg["path"])
    if data_dir is None:
        raise ValueError("Dataset path darf nicht None sein")
    test = np.load(data_dir / "test.npy").astype(np.float32)
    test_phi_path = data_dir / "test_phi.npy"
    test_phi = (
        np.load(test_phi_path).astype(np.float32) if test_phi_path.exists() else None
    )
    context_path = data_dir / "context.npy"
    context_steps_default = (
        int(np.load(context_path).shape[0])
        if context_path.exists()
        else max(1, test.shape[0] // 5)
    )
    metadata_path = data_dir / "metadata.json"
    metadata = (
        json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata_path.exists()
        else {}
    )
    return {
        "name": dataset_cfg.get("name", data_dir.name),
        "path": data_dir,
        "test": test,
        "test_phi": test_phi,
        "context_steps_default": context_steps_default,
        "metadata": metadata,
    }


def selected_dataset_view(
    loaded: dict[str, Any], dataset_cfg: dict[str, Any], trajectory_idx: int
):
    test = loaded["test"]
    test_phi = loaded["test_phi"]
    if test.ndim != 3:
        raise ValueError(f"Expected test.npy shape (T, S, D), got {test.shape}")
    if not (0 <= trajectory_idx < test.shape[1]):
        raise IndexError(
            f"trajectory_idx {trajectory_idx} ausserhalb von [0, {test.shape[1] - 1}]"
        )

    context_start = int(dataset_cfg.get("context_start", 0) or 0)
    context_steps = dataset_cfg.get("context_steps", None)
    context_steps = (
        loaded["context_steps_default"] if context_steps is None else int(context_steps)
    )
    forecast_steps = dataset_cfg.get("forecast_steps", None)

    if context_start < 0:
        raise ValueError("context_start muss >= 0 sein")
    if context_steps <= 0:
        raise ValueError("context_steps muss > 0 sein")
    forecast_start = context_start + context_steps
    if forecast_start >= test.shape[0]:
        raise ValueError(
            f"context_start + context_steps muss < T sein, got {forecast_start} >= {test.shape[0]}"
        )
    end = (
        test.shape[0]
        if forecast_steps is None
        else min(test.shape[0], forecast_start + int(forecast_steps))
    )
    if end <= forecast_start:
        raise ValueError("forecast_steps muss mindestens 1 Punkt erlauben")

    # DynaMix-Forecaster erwartet Batch-Dimension S. Wir geben eine einzelne Trajektorie als S=1 weiter.
    test_view = test[context_start:end, trajectory_idx : trajectory_idx + 1, :]
    if test_phi is not None:
        test_phi_view = test_phi[
            context_start:end, trajectory_idx : trajectory_idx + 1, :
        ]
    else:
        test_phi_view = np.zeros((test_view.shape[0], 1, 1), dtype=np.float32)
    return test_view, test_phi_view, context_steps


@torch.no_grad()
def forecast_local(
    model,
    forecaster,
    test_view: np.ndarray,
    test_phi_view: np.ndarray,
    context_steps: int,
    device: str,
) -> np.ndarray:
    return slides.forecast_once(
        model, forecaster, test_view, test_phi_view, context_steps, device
    )[:, 0, :]


@torch.no_grad()
def forecast_pretrained(
    model, forecaster, test_view: np.ndarray, context_steps: int, device: str
) -> np.ndarray:
    horizon = test_view.shape[0] - context_steps
    context_t = torch.tensor(test_view[:context_steps, :, :], device=device)
    kwargs: dict[str, Any] = {}
    forecast_params = inspect.signature(forecaster.forecast).parameters
    if (
        HF_FORECAST_PREPROCESSING_METHOD is not None
        and "preprocessing_method" in forecast_params
    ):
        kwargs["preprocessing_method"] = HF_FORECAST_PREPROCESSING_METHOD
    if HF_STANDARDIZE is not None and "standardize" in forecast_params:
        kwargs["standardize"] = HF_STANDARDIZE
    pred = forecaster.forecast(context_t, horizon, **kwargs)
    return pred.detach().cpu().numpy().astype(np.float32)[:, 0, :]


def forecast_any(
    spec, model, forecaster, test_view, test_phi_view, context_steps, device
):
    if isinstance(spec, PretrainedSpec):
        return forecast_pretrained(model, forecaster, test_view, context_steps, device)
    return forecast_local(
        model, forecaster, test_view, test_phi_view, context_steps, device
    )


def downsample_indices(n: int, max_points: int | None) -> np.ndarray:
    if max_points is None or n <= max_points:
        return np.arange(n)
    return np.unique(np.linspace(0, n - 1, max_points).round().astype(int))


def make_trajectory_plot(
    truth_full: np.ndarray, pred_future: np.ndarray, context_steps: int, title: str
):
    if truth_full.shape[1] < 3:
        raise ValueError(
            f"3D-Plot braucht mindestens 3 Dimensionen, got {truth_full.shape[1]}"
        )
    pred_full = np.full_like(truth_full, np.nan, dtype=np.float32)
    pred_full[context_steps : context_steps + pred_future.shape[0], :] = pred_future

    idx = downsample_indices(truth_full.shape[0], PLOT_MAX_POINTS)
    fig = go.Figure()
    fig.add_trace(
        go.Scatter3d(
            x=truth_full[idx, 0],
            y=truth_full[idx, 1],
            z=truth_full[idx, 2],
            mode="lines",
            name="Ground truth",
            line=dict(width=5, color="#1f77b4"),
        )
    )
    pred_idx = idx[idx >= context_steps]
    fig.add_trace(
        go.Scatter3d(
            x=pred_full[pred_idx, 0],
            y=pred_full[pred_idx, 1],
            z=pred_full[pred_idx, 2],
            mode="lines",
            name="Prediction",
            line=dict(width=5, color="#d62728"),
        )
    )
    if SHOW_CONTEXT_POINTS:
        context_idx = idx[idx < context_steps]
        fig.add_trace(
            go.Scatter3d(
                x=truth_full[context_idx, 0],
                y=truth_full[context_idx, 1],
                z=truth_full[context_idx, 2],
                mode="lines",
                name="Context",
                line=dict(width=7, color="#2ca02c"),
            )
        )
    fig.update_layout(
        title=title,
        height=PLOT_HEIGHT,
        scene=dict(
            xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="data"
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(l=0, r=0, t=70, b=0),
    )
    return fig


def rmse(truth: np.ndarray, pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((truth - pred) ** 2)))

## Gefundene Modelle und Daten pruefen

In [3]:
model_specs = build_model_specs()
loaded_datasets = [load_dataset(cfg) for cfg in DATASETS]

print(f"Modelle gefunden: {len(model_specs)}")
for spec in model_specs:
    if isinstance(spec, LocalRunSpec):
        print(
            f"- local | {spec.label}\n  run:  {spec.run_dir}\n  repo: {spec.repo_root}"
        )
    else:
        print(f"- pretrained | {spec.label}\n  repo: {spec.repo_root}")

print(f"\nDatensaetze: {len(loaded_datasets)}")
for cfg, data in zip(DATASETS, loaded_datasets):
    print(
        f"- {data['name']} | test={data['test'].shape}, "
        f"test_phi={None if data['test_phi'] is None else data['test_phi'].shape}, "
        f"default_context_steps={data['context_steps_default']}, "
        f"trajectory_indices={cfg.get('trajectory_indices', [])}"
    )

if not model_specs:
    print(
        "\nHinweis: Trage oben mindestens einen LOCAL_MODEL_PATH ein oder setze USE_PRETRAINED_HF=True."
    )

Modelle gefunden: 0

Datensaetze: 1
- burke_shaw_values_8 | test=(12000, 128, 3), test_phi=(12000, 128, 1), default_context_steps=500, trajectory_indices=[0]

Hinweis: Trage oben mindestens einen LOCAL_MODEL_PATH ein oder setze USE_PRETRAINED_HF=True.


## Forecasts berechnen und Plotly-3D-Plots anzeigen

In [ ]:
if DRY_RUN:
    print(
        "DRY_RUN=True: Es werden keine Modelle geladen und keine Forecasts berechnet."
    )
else:
    effective_device = resolve_device(DEVICE)
    print(f"Device: {effective_device}")
    results = []
    for spec_idx, spec in enumerate(model_specs, start=1):
        print(f"\n[{spec_idx}/{len(model_specs)}] Lade Modell: {spec.label}")
        model, forecaster = load_any_model(spec, effective_device)
        for dataset_cfg, loaded in zip(DATASETS, loaded_datasets):
            for trajectory_idx in dataset_cfg.get("trajectory_indices", []):
                test_view, test_phi_view, context_steps = selected_dataset_view(
                    loaded, dataset_cfg, int(trajectory_idx)
                )
                truth_full = test_view[:, 0, :]
                truth_future = truth_full[context_steps:, :]
                pred_future = forecast_any(
                    spec,
                    model,
                    forecaster,
                    test_view,
                    test_phi_view,
                    context_steps,
                    effective_device,
                )
                score = rmse(truth_future, pred_future)
                title = (
                    f"{spec.label} | {loaded['name']} | traj={trajectory_idx} | "
                    f"context={context_steps} | RMSE={score:.5g}"
                )
                print(title)
                fig = make_trajectory_plot(
                    truth_full, pred_future, context_steps, title
                )
                fig.show()
                results.append(
                    {
                        "model": spec.label,
                        "dataset": loaded["name"],
                        "trajectory_idx": int(trajectory_idx),
                        "context_steps": int(context_steps),
                        "rmse": score,
                        "truth": truth_full,
                        "prediction_future": pred_future,
                    }
                )
        del model, forecaster
        if effective_device.startswith("cuda") and torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"\nFertig. Forecasts im Notebook-Objekt `results`: {len(results)}")